# Antasena Super Course — Belajar MQTT

Di notebook ini kamu akan mencoba sendiri cara kerja **MQTT**, protokol yang dipakai mobil untuk mengirim data telemetri secara langsung (live) ke tim di pinggir lintasan.

**Yang akan kamu lakukan:**
1. Terhubung ke sebuah *broker* MQTT publik.
2. *Subscribe* ke topik milikmu sendiri, lalu *publish* pesan ke sana dan melihatnya datang.
3. Mengirim beberapa baris data telemetri sungguhan, persis seperti yang dilakukan mobil.
4. Mendengarkan siaran langsung dari instruktur selama beberapa saat dan menyimpannya ke CSV.

Notebook ini untuk **memahami MQTT**. Untuk analisis efisiensi, pakai data lengkap yang sudah disediakan di `data/` (lihat `analysis_exercise.ipynb`), karena merekam satu balapan penuh butuh waktu lama.

Jangan khawatir mengubah apa pun: perubahanmu hanya ada di salinanmu sendiri. Simpan lewat **File → Save a copy in Drive** jika ingin menyimpannya.

## Apa itu MQTT?

MQTT adalah cara mengirim pesan kecil lewat internet dengan pola **publish/subscribe**:

```
  mobil (publisher)                                 laptop tim (subscriber)
        |                                                    ^
        |  publish ke topik                                  |  broker meneruskan pesan
        |  "antasena/course/telemetry"                       |  ke semua yang subscribe
        v                                                    |
   +-------------------------- BROKER --------------------------+
   |                   broker.hivemq.com : 1883                 |
   +------------------------------------------------------------+
```

- **Broker**: server perantara. Publisher dan subscriber tidak perlu saling kenal; keduanya cukup terhubung ke broker yang sama.
- **Topik**: "alamat" pesan, berupa teks bertingkat seperti `antasena/course/telemetry`. Subscriber hanya menerima pesan dari topik yang ia subscribe.
- **Publish**: mengirim pesan ke sebuah topik.
- **Subscribe**: mendaftar untuk menerima semua pesan baru di sebuah topik. Pesan yang dikirim *sebelum* kamu subscribe tidak akan kamu terima.
- **QoS** (Quality of Service): tingkat jaminan pengiriman. QoS 0 = kirim sekali tanpa konfirmasi, QoS 1 = dijamin sampai minimal sekali.

Di kursus ini, setiap pesan berisi satu baris data telemetri dalam format JSON, misalnya `{"lap": 1, "time_s": 12.4, "voltage_V": 104.9, "current_A": 3.2, ...}`.

## Persiapan

In [ ]:
import os, sys

if "google.colab" in sys.modules and not os.path.exists("data"):
    if not os.path.exists("/content/ASC_Simulator"):
        !git clone -q https://github.com/Antasena-ITS-Team/ASC_Simulator.git /content/ASC_Simulator
    %cd /content/ASC_Simulator

In [ ]:
!pip install -q paho-mqtt

import json, re, time
import pandas as pd
import paho.mqtt.client as mqtt


def tunggu(client, detik):
    """Biarkan client memproses pesan masuk/keluar selama beberapa detik."""
    batas = time.time() + detik
    while time.time() < batas:
        client.loop(timeout=0.1)

## Langkah 1 — Pengaturan koneksi

Untuk terhubung ke MQTT, kita butuh tiga hal:

| Pengaturan | Nilai | Artinya |
|---|---|---|
| `BROKER` | `broker.hivemq.com` | alamat server broker. Ini broker publik gratis milik HiveMQ, siapa pun bisa memakainya tanpa akun. |
| `PORT` | `1883` | nomor "pintu" di server. 1883 adalah port standar MQTT tanpa enkripsi (8883 untuk MQTT terenkripsi/TLS). |
| `TOPIK_KELAS` | `antasena/course/telemetry` | topik tempat instruktur menyiarkan data mobil. Harus **sama persis** dengan topik di `publisher.py`. |

Karena broker ini publik, **siapa pun bisa membaca dan mengirim ke topik mana pun**. Jangan mengirim data rahasia lewat broker publik; tim sungguhan memakai broker sendiri dengan username/password.

✏️ Isi `NAMA` dengan namamu (tanpa spasi). Namamu dipakai untuk membuat topik pribadi supaya pesanmu tidak tercampur dengan peserta lain.

In [ ]:
BROKER = "broker.hivemq.com"                # alamat broker
PORT = 1883                                 # port standar MQTT
TOPIK_KELAS = "antasena/course/telemetry"   # topik siaran instruktur

NAMA = "nama_kamu"                          # ✏️ ganti dengan namamu
TOPIK_SAYA = "antasena/course/latihan/" + re.sub(r"[^a-z0-9_]", "", NAMA.lower())

print(f"Broker      : {BROKER}:{PORT}")
print(f"Topik kelas : {TOPIK_KELAS}")
print(f"Topik saya  : {TOPIK_SAYA}")

### Membuat client dan terhubung

Client MQTT memakai **callback**: fungsi yang dipanggil otomatis saat sesuatu terjadi.
- `on_connect` dipanggil saat broker menerima koneksi kita.
- `on_message` dipanggil setiap kali ada pesan masuk dari topik yang kita subscribe. Di sini setiap pesan disimpan ke daftar `pesan_masuk`.

Lalu `client.connect(BROKER, PORT)` membuka koneksi ke broker. `keepalive=60` artinya client mengirim sinyal "masih hidup" setiap 60 detik agar koneksi tidak diputus.

In [ ]:
pesan_masuk = []


def on_connect(client, userdata, flags, reason_code, properties):
    print("Terhubung ke broker!" if not reason_code.is_failure else f"Gagal terhubung: {reason_code}")


def on_message(client, userdata, msg):
    pesan_masuk.append((msg.topic, msg.payload.decode("utf-8")))


client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
client.on_connect = on_connect
client.on_message = on_message
print(f"Menghubungkan ke {BROKER}:{PORT} ...")
client.connect(BROKER, PORT, keepalive=60)
tunggu(client, 2)

## Langkah 2 — Subscribe, lalu kirim pesan ke diri sendiri

Kita subscribe ke topik pribadi, lalu publish pesan ke topik yang sama. Pesan itu pergi ke broker di internet dan kembali lagi ke kita karena kita sedang subscribe.

In [ ]:
client.subscribe(TOPIK_SAYA, qos=1)
tunggu(client, 1)

pesan_masuk.clear()
client.publish(TOPIK_SAYA, f"Halo dari {NAMA}!", qos=1)
tunggu(client, 2)

for topik, isi in pesan_masuk:
    print(f"Diterima di [{topik}]: {isi}")

**Coba sendiri:**
- Ubah isi pesannya lalu jalankan ulang sel di atas.
- Publish ke topik lain (misalnya `TOPIK_SAYA + "/lain"`). Apakah pesannya diterima? Mengapa tidak?
- Minta temanmu publish ke topik pribadimu dari notebook mereka.

## Langkah 3 — Jadi "mobil": kirim data telemetri

Sekarang kita meniru `publisher.py`: ambil 20 baris dari data rekaman, ubah setiap baris menjadi JSON, dan publish satu per satu. Di sisi penerima, JSON diubah kembali menjadi tabel.

In [ ]:
rekaman = pd.read_csv("data/DataASC_105V_9A.csv").iloc[1000:1020]

pesan_masuk.clear()
for _, baris in rekaman.iterrows():
    client.publish(TOPIK_SAYA, json.dumps(baris.to_dict()), qos=1)
    tunggu(client, 0.1)
tunggu(client, 2)

print(f"Terkirim {len(rekaman)} pesan, diterima {len(pesan_masuk)} pesan")
print("Contoh pesan mentah:", pesan_masuk[0][1] if pesan_masuk else "-")

diterima = pd.DataFrame([json.loads(isi) for _, isi in pesan_masuk])
diterima

## Langkah 4 — Dengarkan siaran langsung instruktur

Instruktur menjalankan `publisher.py` yang menyiarkan data balapan ke `TOPIK_KELAS`. Jalankan sel di bawah **saat instruktur memberi aba-aba**. Sel ini mendengarkan selama `DURASI_S` detik, menampilkan jumlah pesan yang masuk, lalu menyimpannya ke CSV.

Jika jumlah pesan tetap 0, berarti belum ada yang menyiarkan ke topik itu. Ingat: pesan yang dikirim sebelum kamu subscribe tidak akan kamu terima.

In [ ]:
DURASI_S = 60                  # berapa detik mendengarkan
OUT_FILE = "rekaman_saya.csv"

client.subscribe(TOPIK_KELAS, qos=1)
pesan_masuk.clear()
print(f"Mendengarkan {TOPIK_KELAS} selama {DURASI_S} detik...")
for detik in range(0, DURASI_S, 10):
    tunggu(client, min(10, DURASI_S - detik))
    jumlah = sum(1 for topik, _ in pesan_masuk if topik == TOPIK_KELAS)
    print(f"  {detik + min(10, DURASI_S - detik):>4} s: {jumlah} pesan diterima")
client.unsubscribe(TOPIK_KELAS)

baris = [json.loads(isi) for topik, isi in pesan_masuk if topik == TOPIK_KELAS]
if baris:
    siaran = pd.DataFrame(baris)
    siaran.to_csv(OUT_FILE, index=False)
    print(f"Tersimpan {len(siaran)} baris ke {OUT_FILE}")
    display(siaran.tail())
else:
    print("Tidak ada pesan. Apakah instruktur sedang menjalankan publisher.py?")

**Pertanyaan:**
1. Berapa pesan per detik yang kamu terima? Bandingkan dengan kolom `time_s`: apakah data diputar lebih cepat dari waktu aslinya?
2. Jika dua peserta subscribe ke topik yang sama, apakah keduanya menerima pesan yang sama?
3. Mengapa pola publish/subscribe cocok untuk telemetri mobil balap, dibandingkan mobil mengirim data langsung ke satu laptop?

## Langkah 5 — Putuskan koneksi dan unduh hasil

File di Colab hilang saat runtime berakhir, jadi unduh CSV-mu jika ingin menyimpannya.

In [ ]:
client.disconnect()
print("Koneksi ditutup.")

import os, sys
if "google.colab" in sys.modules and os.path.exists(OUT_FILE):
    from google.colab import files
    files.download(OUT_FILE)

## Selanjutnya

Lanjutkan ke `analysis_exercise.ipynb` untuk menganalisis efisiensi memakai data balapan lengkap yang sudah disediakan di `data/`.